In [20]:
from gradescope_auth import create_new_user, login_with_token, login_temporary, save_profile_for_token
from concurrent.futures import ThreadPoolExecutor
%load_ext autoreload 
%autoreload 2 

token = 'jzGzeA1Z7e_izljQXT4Mki4NsxUTKOhrqD30L-iwtTs'

if token:
    with ThreadPoolExecutor(max_workers=1) as executor:
        conn = executor.submit(lambda: login_with_token(token)).result()
else:
    with ThreadPoolExecutor(max_workers=1) as executor:
        conn, temp_profile_dir = executor.submit(login_temporary).result()
    token = create_new_user()
    save_profile_for_token(temp_profile_dir, token)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [125]:
courses = conn.account.get_courses()
# course_id = [k for (k,v) in courses['instructor'].items() if v.name=='COSI 21a'][0]
course_id = [k for (k,v) in courses['instructor'].items() if v.name=='TEST'][0]
assignment_id = conn.account.get_assignments(course_id)[-1].assignment_id

course_id, assignment_id

('1311586', '8237331')

In [126]:
from utils import * 
students, max_student_name_length = get_student_info(conn, course_id)
instructors = get_instructor_info(conn, course_id)
# student_mapping = get_user_mapping(students)
# instructor_mapping = get_user_mapping(instructors)
users = students + instructors
# user_mapping = student_mapping | instructor_mapping
questions, questions_order = get_assignment_questions(conn, course_id, assignment_id)
raw_submissions_metadata = get_raw_submissions_metadata(conn, course_id, assignment_id)
grades_metadata = get_grades_metadata(conn, course_id, assignment_id, instructors, users)
student_to_assignment_submissions = get_student_to_assignment_submissions(users, raw_submissions_metadata, grades_metadata)
grader_by_question_submission = get_grader_by_question_submission(conn, course_id, questions)
question_to_submissions = get_question_to_question_submissions(conn, course_id, questions)
comments, total_scores, student_to_question_to_question_submission = get_raw_data_by_question_submission(conn, course_id, users, questions, question_to_submissions, student_to_assignment_submissions)
grade_breakdowns = get_grade_breakdowns(users, questions, comments, total_scores, student_to_question_to_question_submission, grader_by_question_submission, questions_order)
users_with_grades = [u for u in users if grades_metadata[u.email_address]['submitted']]


In [ ]:
from collections import defaultdict
from statistics import mean, median
import pandas as pd


def get_assignment_summary(questions, questions_order, grade_breakdowns, users_with_grades,):
    def outline(question):
        path = [question.title]
        p = question.parent
        while p is not None:
            path = [p.title] + path
            p = p.parent
        return '\n'.join(["--"*i+p for (i,p) in enumerate(path)])
    def rubric_string(question):
        if not question.rubric_items:
            return ""
        items = list(question.rubric_items.values())
        groups = {}
        ungrouped = []
        for item in items:
            if item.rubric_group_id is None:
                ungrouped.append(item)
            else:
                groups.setdefault(
                    item.rubric_group_id,
                    {
                        "description": item.rubric_group_description,
                        "items": [],
                    },
                )["items"].append(item)
        def has_points(group):
            return group.points != 0 if isinstance(group, RubricItem) else any(item.points != 0 for item in group["items"])
        def format_item(item, indent=0):
            if item.points > 0:
                prefix = f"+{item.points:g} "
            elif item.points < 0:
                prefix = f"{item.points:g} "
            else:
                prefix = "+0 " if question.scoring_type == "positive" else f"{BULLETS[indent % len(BULLETS)]} "
            return "    " * indent + prefix + item.description
        # Stable partition: scored things first
        group_list = list(groups.values())
        all_items = ungrouped + group_list
        all_items.sort(key=lambda g: not has_points(g))
        lines = []
        for item in all_items:
            if isinstance(item, RubricItem):
                lines.append(format_item(item))
            else:
                lines.append(f"{BULLETS[0]} {item['description']}")
                item["items"].sort(key=lambda i: i.points == 0)
                for item in item["items"]:
                    lines.append(format_item(item, indent=1))
        return "\n".join(lines)
    def stats_string(scores):
        scores = [x for x in scores if x is not None]
        if not scores:
            return ""
        return f"Count: {len(scores)}\nMean: {mean(scores):.2f}\nMedian: {median(scores):.2f}"
    def grader_stats_string(grader_scores):
        pieces = []
        for grader in sorted(grader_scores):
            scores = [x for x in grader_scores[grader] if x is not None]
            if not scores:
                continue
            pieces.append(f"{grader}\n  Count: {len(scores)}\n  Mean: {mean(scores):.2f}\n  Median: {median(scores):.2f}")
        return "\n\n".join(pieces)
    rows = []
    for question in questions_order:
        scores = []
        grader_scores = defaultdict(list)
        for user in users_with_grades:
            if not any(g.question_id == question for g in grade_breakdowns[user.identifier]):
                continue
            breakdown = [g for g in grade_breakdowns[user.identifier] if g.question_id == question][0]
            score = breakdown.total_score
            grader = breakdown.grader
            scores.append(score)
            grader_name = (
                grader.full_name
                if hasattr(grader, "full_name")
                else getattr(grader, "email_address", str(grader))
            )
            grader_scores[grader_name].append(score)
        rows.append(
            {
                "Question": questions[question].title,
                "Outline": outline(questions[question]),
                "Max Points": questions[question].max_grade,
                "Rubric": rubric_string(questions[question]),
                "Stats": stats_string(scores),
                "Stats by grader": grader_stats_string(grader_scores),
            }
        )
    return pd.DataFrame(rows)


pd.set_option('display.max_colwidth', None)

df = get_assignment_summary(questions, questions_order, grade_breakdowns, users_with_grades)
df.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'})


,Question,Outline,Max Points,Rubric,Stats,Stats by grader
0,Question1,Question1,6.000000,• Correct,,
1,Q1 part1,Question1 --Q1 part1,1.000000,"-0.25 This is a comment that isn't linked to a specific part of the page; it's just a rubric item within Q1.1 -0.25 Here's a comment that is worth 0.25 points and has _some_ **formatting** in it! • This is a rubric item group! -0.4 This is a comment inside of the group -0.3 This is another comment inside of the group • This is also a rubric item but this one isn't worth any points, it's just a comment • And here's one more that _is_ linked to a point on the page for student 1, but not for other students. • This is one that is linked to a point on the page for all students.",Count: 4 Mean: 0.50 Median: 0.50,Ruth Rosenblum Count: 4 Mean: 0.50 Median: 0.50
2,Q1 part2,Question1 --Q1 part2,2.000000,"-2 And this one, but this is worth the full 2.0 points -0.2 And here's a smaller one linked to a place in the text • This is a Q1.2 comment!",Count: 4 Mean: 0.95 Median: 0.90,Ruth Rosenblum Count: 4 Mean: 0.95 Median: 0.90
3,Q1 part3,Question1 --Q1 part3,3.000000,"-0.2 This is a larger part of Q1 so it's going to have more comments and rubric items -2 This is a big deduction comment -1 This is one notch worth of points • Correct • This is a comment saying something is wrong, but it's not deducting any points",Count: 4 Mean: 2.75 Median: 3.00,Ruth Rosenblum Count: 4 Mean: 2.75 Median: 3.00
4,Question2,Question2,3.000000,+1 This is one part of the correct answer +2 This is another part of the correct answer +1 This is an alternative to the second part but it's only worth half the points +0 Correct,Count: 4 Mean: 1.75 Median: 2.00,Ruth Rosenblum Count: 4 Mean: 1.75 Median: 2.00
